# 9 — Prvi poziv LLM-a

**Četvrtak, 14:00.** Drugi dio popodnevnog bloka.

U 13h smo YOLO zapakirali u `detect()`. Sad radimo isto s LLM-om: jedan
poziv, pa funkcija `ask()`.

Namjerno **bez SDK-a**, običnim HTTP POST-om. Poanta je da vidite da LLM API
nije magija nego JSON preko HTTP-a — to će vam trebati u 15h kad budemo
gradili workflow, i u 16h kad agent sam bude odlučivao što zvati.

Dva pitanja koja želim da odgovorite do kraja notebooka:
1. Koliko je koštalo?
2. Kako znate da je odgovor točan?

In [ ]:
# --- SETUP: pokreni ovo prvo ---  [lares-setup-v1]
# Radi i u Colabu i lokalno. Sigurno je pokrenuti vise puta.
import os, sys, subprocess
REPO = "ai_bootcamp_foundations"
if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/hrvojenovak/{REPO}.git"],
                       cwd="/content", check=True)
    os.chdir(f"/content/{REPO}/notebooks")
print("cwd:", os.getcwd(), "| data ok:", os.path.isdir("../data"))

## 1. API ključ

**Svaki od vas treba svoj ključ.** Kvota se broji po projektu, ne po ključu,
pa dijeljeni ključ znači dijeljenu kvotu.

1. Otvorite [aistudio.google.com/apikey](https://aistudio.google.com/apikey)
2. **Create API key** → kopirajte
3. U Colabu lijevo kliknite **ključić 🔑** (Secrets)
4. Novi secret: ime `GOOGLE_API_KEY`, vrijednost = vaš ključ
5. Uključite **Notebook access**

Ako AI Studio kaže da nije dostupan: koristite **osobni** Google račun, ne
poslovni. Firmini Workspace računi često imaju to blokirano.

In [ ]:
API_KEY = None

if "google.colab" in sys.modules:
    from google.colab import userdata
    try:
        API_KEY = userdata.get("GOOGLE_API_KEY")
    except Exception as e:
        print("Secret nije dostupan:", type(e).__name__)
else:
    API_KEY = os.environ.get("GOOGLE_API_KEY")

if API_KEY:
    print(f"kljuc ucitan: {API_KEY[:6]}...{API_KEY[-4:]}  ({len(API_KEY)} znakova)")
else:
    print("NEMA KLJUCA — vrati se na korak 1-5 iznad.")
    print("Ne lijepi kljuc direktno u celiju: notebook ide na GitHub.")

## 2. Goli HTTP poziv

Ovo je *cijeli* API. Jedan endpoint, jedan JSON.

`contents` je razgovor. Role su `user` i `model` (kod Gemini API-ja; drugi
provideri koriste `assistant` — interface se razlikuje, ideja ne).

In [ ]:
import requests, json

MODEL = "gemini-2.5-flash"     # imena se mijenjaju; ako dobijes 404, provjeri u AI Studiju
URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"

payload = {
    "contents": [
        {"role": "user", "parts": [{"text": "Objasni u jednoj rečenici što je overfitting."}]}
    ],
    "generationConfig": {"temperature": 0.0, "maxOutputTokens": 200},
}

resp = requests.post(URL, params={"key": API_KEY}, json=payload, timeout=60)
print("HTTP", resp.status_code)
data = resp.json()
print(json.dumps(data, indent=1, ensure_ascii=False)[:900])

## 3. `ask()` — interface za ljude i za kod

Isti obrazac kao `detect()` u YOLO notebooku: zamotaj u funkciju s čistim
potpisom. Dodajemo dvije stvari koje ćete stvarno trebati.

**Retry na 429.** Free tier ima ~10-15 zahtjeva u minuti. 429 nije greška,
to je normalno stanje. Backoff ima *jitter* jer nas je trinaest u sobi — bez
njega svi retryjamo u istoj sekundi i sami držimo limit probijenim.

**Brojač.** Dnevna kvota je ~250-1500 zahtjeva, ovisno o modelu. Agent u 16h
pojede 10+ zahtjeva po pokretanju, pa je dobro znati gdje ste.

In [ ]:
import time, random

STATS = {"requests": 0, "input_tokens": 0, "output_tokens": 0, "thought_tokens": 0}


def _post(body: dict, model: str, max_attempts: int = 5) -> dict:
    """POST s retryjem. Vraca raw JSON ili {"_error": "..."}."""
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
    for attempt in range(1, max_attempts + 1):
        r = requests.post(url, params={"key": API_KEY}, json=body, timeout=90)

        # 429 zna znaciti dvije razlicite stvari - razlikujemo ih
        if r.status_code == 429 and "billing" in r.text.lower():
            return {"_error": "KVOTA/BILLING: ovo nije rate limit. "
                              "Grounding nije dostupan na free tieru API-ja."}

        if r.status_code in (429, 500, 502, 503, 504):
            if attempt == max_attempts:
                return {"_error": f"odustajem nakon {attempt} pokusaja: HTTP {r.status_code}"}
            delay = min(32, 2 ** (attempt - 1)) * (0.5 + random.random())   # jitter
            print(f"  HTTP {r.status_code}, cekam {delay:.1f}s (pokusaj {attempt})")
            time.sleep(delay)
            continue

        if r.status_code >= 400:
            return {"_error": f"HTTP {r.status_code}: {r.text[:300]}"}
        return r.json()
    return {"_error": "neocekivano"}


def _parse(d: dict) -> dict:
    """Izvuci tekst, izvore i finishReason iz odgovora."""
    if "_error" in d:
        return {"text": f"[{d['_error']}]", "sources": [], "queries": [],
                "finish": "ERROR", "ok": False}

    u = d.get("usageMetadata", {})
    STATS["requests"] += 1
    STATS["input_tokens"] += u.get("promptTokenCount", 0)
    STATS["output_tokens"] += u.get("candidatesTokenCount", 0)
    STATS["thought_tokens"] += u.get("thoughtsTokenCount", 0)

    cands = d.get("candidates") or []
    if not cands:
        return {"text": "[nema odgovora - vjerojatno safety filter]", "sources": [],
                "queries": [], "finish": "NO_CANDIDATES", "ok": False}

    c = cands[0]
    text = "".join(x.get("text", "") for x in (c.get("content") or {}).get("parts", []))
    finish = c.get("finishReason", "?")

    meta = c.get("groundingMetadata") or {}
    sources = [{"title": (ch.get("web") or {}).get("title", "?"),
                "uri": (ch.get("web") or {}).get("uri", "")}
               for ch in (meta.get("groundingChunks") or []) if ch.get("web")]

    return {"text": text.strip(), "sources": sources,
            "queries": meta.get("webSearchQueries") or [],
            "finish": finish, "ok": True}


def ask(prompt: str, *, model: str = MODEL, temperature: float = 0.0,
        max_tokens: int = 4096, thinking: bool = False) -> str:
    """Posalji prompt, vrati tekst.

    thinking=False iskljucuje 'razmisljanje'. Gemini 2.5 ga ima UKLJUCENO po
    defaultu, a te tokene placa iz istog maxOutputTokens budzeta - zato ti se
    odgovor odreze na pola recenice iako si dao velik limit.
    """
    body = {
        "contents": [{"role": "user", "parts": [{"text": prompt}]}],
        "generationConfig": {"temperature": temperature, "maxOutputTokens": max_tokens},
    }
    if not thinking:
        body["generationConfig"]["thinkingConfig"] = {"thinkingBudget": 0}

    res = _parse(_post(body, model))
    if res["finish"] == "MAX_TOKENS":
        return res["text"] + "\n\n[ODREZANO: finishReason=MAX_TOKENS -> povecaj max_tokens]"
    return res["text"]


print(ask("Nabroji tri nacina da model pretreniras. Kratko."))
print("\n", STATS)

## 4. Tokenizacija — što točno plaćate

Do sad ste vidjeli brojeve tokena, ali ne i **što je token**. Bez toga
`input_tokens` je samo broj s računa.

Tri stvari koje ova sekcija pokazuje: kako se tekst reže na tokene, zašto
hrvatski košta više od engleskog, i zašto brojke iz tablica koštaju najviše.

In [ ]:
# countTokens je ZASEBAN endpoint - vraca tocan broj tokena BEZ generiranja
def count_tokens(text: str, model: str = MODEL) -> int:
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:countTokens"
    r = requests.post(url, params={"key": API_KEY},
                      json={"contents": [{"role": "user", "parts": [{"text": text}]}]},
                      timeout=60)
    if r.status_code >= 400:
        return -1
    return r.json().get("totalTokens", -1)


hr = "Instalirana snaga vjetroelektrana na dan 31.12.2025. iznosila je 1 347,8 MW."
en = "The installed wind power capacity as of 31 December 2025 was 1,347.8 MW."

for label, t in [("hrvatski", hr), ("engleski", en)]:
    n = count_tokens(t)
    print(f"  {label}: {len(t):3d} znakova -> {n:3d} tokena "
          f"({len(t)/n:.2f} znakova/token)" if n > 0 else f"  {label}: greska")

### Kako izgleda rezanje

`countTokens` daje broj, ali ne pokazuje **granice**. Za to koristimo
`tiktoken` — tokenizer OpenAI-jevih modela.

Nije isti tokenizer kao Geminijev, pa se brojevi neće poklopiti. **I to je
lekcija:** tokenizer je dio modela, ne univerzalno pravilo. Ali mehanizam
je isti, a granice se vide.

In [ ]:
%pip install -q tiktoken
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

def show_tokens(text: str) -> None:
    ids = enc.encode(text)
    toks = [enc.decode([i]) for i in ids]
    print(f"  {len(ids):3d} tokena | " + " · ".join(t.replace(" ", "␣") for t in toks))

for t in [hr, en, "1 347,8 MW", "1347.8 MW", "vjetroelektrana", "wind farm",
          "sunčana", "suncana"]:
    print(f"\n{t!r}"); show_tokens(t)

Tri stvari koje se u tom ispisu vide:

- **Brojke s decimalnim zarezom** pucaju na pojedinačne cifre. `1 347,8` je
  skuplji od `1347.8`, koji je skuplji od `1348`.
- **Dijakritici** dodaju tokene: `sunčana` protiv `suncana`.
- **Razmaci pripadaju tokenu** koji slijedi (` MW` je jedan token, ne dva).

To je razlog zašto vam je dokument iz notebooka 10 skuplji od procjene
`len(tekst)/4` — ta procjena vrijedi za engleski.

### Zašto to tako radi: BPE u 20 linija

Tokenizeri se **uče iz podataka** algoritmom BPE (byte pair encoding):
počni od pojedinačnih znakova i uzastopno spajaj najčešći susjedni par.

Pokrenite ovo i pogledajte korake. Nema API poziva, čista logika.

In [ ]:
from collections import Counter

def train_bpe(words, n_merges=12):
    vocab = {" ".join(w) + " </w>": c for w, c in Counter(words).items()}
    merges = []
    for step in range(n_merges):
        pairs = Counter()
        for word, freq in vocab.items():
            sym = word.split()
            for i in range(len(sym) - 1):
                pairs[(sym[i], sym[i+1])] += freq
        if not pairs:
            break
        best, freq = pairs.most_common(1)[0]
        merges.append(best)
        print(f"  {step+1:2d}. {best[0]!r} + {best[1]!r} -> {(best[0]+best[1])!r}   (pojava: {freq})")
        vocab = {w.replace(" ".join(best), "".join(best)): c for w, c in vocab.items()}
    return merges


def apply_bpe(word, merges):
    sym = list(word) + ["</w>"]
    for a, b in merges:
        i = 0
        while i < len(sym) - 1:
            if sym[i] == a and sym[i+1] == b:
                sym[i:i+2] = [a + b]
            else:
                i += 1
    return sym


corpus = ("elektrana elektrane elektranu elektrana vjetroelektrana vjetroelektrane "
          "hidroelektrana hidroelektrane snaga snage snagu instalirana instalirane "
          "elektrana elektrane snaga vjetroelektrana").split()

print("Ucim BPE na malom hrvatskom korpusu:\n")
merges = train_bpe(corpus)

print("\nPrimjena:")
for w in ["elektrana", "vjetroelektrana", "sunčana", "1347,8"]:
    t = apply_bpe(w, merges)
    print(f"  {w:18s} -> {t}  ({len(t)} tokena)")

Pogledajte ishod. `elektrana` je **jedan** token jer je u korpusu česta.
`vjetroelektrana` je šest, jer prefiks nije naučen. A `1347,8` je sedam —
BPE nikad nije vidio dovoljno cifara da ih spoji.

Pravi tokenizeri su učeni na tekstu koji je **pretežno engleski**. Zato
hrvatske riječi pucaju na sitno, a brojevi iz tablica najviše od svega.
Vaš RAG nad hrvatskim izvještajem plaća upravo to.

### Koliko to košta

Cijene su po milijun tokena. Ono što ljudi promaše: u razgovoru se **cijela
povijest šalje ponovno u svakom pozivu**, pa trošak raste kvadratno.

In [ ]:
PRICES = {"Gemini Flash (paid)": (0.30, 2.50), "Claude Haiku 4.5": (1, 5),
          "Claude Sonnet 5": (2, 10)}

i, o = STATS["input_tokens"], STATS["output_tokens"]
print(f"do sad: {STATS['requests']} zahtjeva, {i:,} input + {o:,} output tokena\n")
for name, (pin, pout) in PRICES.items():
    print(f"  {name:22s} ${i*pin/1e6 + o*pout/1e6:.6f}")

print("\nRazgovor od 10 izmjena, po 800 tokena svaka:")
turn, total = 800, 0
for t in range(1, 11):
    total += turn * t                       # povijest se ponovno salje
    if t in (1, 5, 10):
        print(f"  nakon {t:2d}. izmjene: {total:>7,} input tokena kumulativno")
print(f"\n-> 10 izmjena nije 10x jedna, nego {total // turn}x.")

## 5. Grounding s izvorima

> **Pazi: grounding NIJE dostupan na free tieru API-ja.** Besplatan je samo u
> samom AI Studio *sucelju*. Ako preko API-ja dobijes HTTP 429 s tekstom o
> *plan and billing details* — to nije rate limit i nema smisla cekati.
> Kako to onda vidjeti: pogledaj sekciju 6.

`ask()` vraca samo tekst i **baca izvore**. To nije dobro: grounded odgovor
bez izvora je isto tako neprovjerljiv kao ungrounded.

Kad ukljucis Google Search, Gemini uz odgovor vraca i `groundingMetadata`:
koje je upite pretrazio i koje stranice je koristio. Izvucimo to.

In [ ]:
def ask_grounded(prompt: str, model: str = MODEL, max_tokens: int = 4096) -> dict:
    """Grounded poziv koji vraca I odgovor I izvore. Isti retry kao ask()."""
    body = {
        "contents": [{"role": "user", "parts": [{"text": prompt}]}],
        "tools": [{"google_search": {}}],
        "generationConfig": {"temperature": 0.0, "maxOutputTokens": max_tokens},
    }
    return _parse(_post(body, model))


def show(res: dict) -> None:
    print(res["text"], "\n")

    if not res["ok"]:
        print("POZIV NIJE PROSAO - gore je uzrok. Ne znaci da je model odgovorio iz tezina.")
        return

    if res["queries"]:
        print("pretrazeni upiti:", res["queries"])
    if res["sources"]:
        print(f"izvori ({len(res['sources'])}):")
        for s in res["sources"]:
            print(f"  - {s['title']}")
    else:
        print("BEZ IZVORA -> poziv je prosao, ali model NIJE pretrazivao. "
              "Odgovorio je iz tezina.")

### Isti upit, dva puta

Postavite pitanje iz **svoje** domene gdje znate tocan odgovor. Najbolje rade
pitanja o **konkretnom broju ili datumu**, jer opca pitanja model odgovori
dovoljno neodredeno da razlike ne vidite.

Tri stvari koje gledajte, ne dvije:

1. Je li ungrounded odgovor tocan?
2. Je li grounded odgovor tocan?
3. **Jesu li izvori dobri?** Ako je model nasao blog iz 2019., grounded
   odgovor je *provjerljivo* pogresan — a to je bolje od nepovjerljivo
   pogresnog, ali nije tocno.

Ako se ispise "BEZ IZVORA", model je odlucio da ne treba pretrazivati.
To je isto rezultat: grounding je *dostupan* alat, ne *obavezan* korak.

## 6. Grounded vs ungrounded — glavna poanta

Isto pitanje, dva puta. Bez alata model odgovara **iz težina** — iz onoga što
je zapamtio do svog knowledge cutoffa. S `grounded=True` dobiva Google Search
kao alat i može provjeriti.

Postavite pitanje iz **svoje** domene gdje znate točan odgovor. Najbolje
rade pitanja o **konkretnom broju ili datumu**, jer opća pitanja model
odgovori dovoljno neodređeno da razlike ne vidite.

In [ ]:
PITANJE = "Koliki je bio instalirani kapacitet vjetroelektrana u Hrvatskoj?"

print("=== BEZ alata (iz tezina) ===")
print(ask(PITANJE), "\n")

print("=== S Google Searchom (grounded) ===")
show(ask_grounded(PITANJE))

print("\n", STATS)

## 7. Za zapamtiti

**LLM API je HTTP POST s JSON-om.** Nema magije. SDK je udobnost, ne
nužnost — i zato ćete u 16h moći napisati agenta u 50 linija.

**Grounding je tool use, ne svojstvo modela.** Web search nije "opcija
modela", to je alat koji mu dodaš. Isto kao `detect()` iz 13h.

**Grounding premješta problem, ne rješava ga.** Model sad može naći *loš*
izvor i uvjereno ga citirati. Ako je ungrounded odgovor bio neodređen a
grounded konkretan — provjerite je li konkretan i *točan*.

**Tokeni se broje.** Povijest razgovora se ponovno šalje u svakom pozivu.

---

### Ako nešto ne radi

| simptom | uzrok |
|---|---|
| `NEMA KLJUCA` | secret nije dodan, ili Notebook access nije uključen |
| HTTP 400 | ime modela — provjerite u AI Studiju |
| HTTP 403 | ključ nevažeći, ili AI Studio blokiran na računu |
| HTTP 429 | rate limit — `ask()` sam retryja, samo pričekajte |
| `[nema odgovora]` | safety filter — preformulirajte prompt |